In [ ]:
import os
import pandas as pd
import torch
from datasets import Dataset
from transformers import AutoModelForCausalLM, AutoTokenizer, Trainer, TrainingArguments, DataCollatorForLanguageModeling
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")
MODEL_NAME = "microsoft/DialoGPT-small"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.pad_token = tokenizer.eos_token 
data_path = "/kaggle/working/processed/train_pairs.csv" 
df = pd.read_csv(data_path).dropna(subset=['prompt', 'response'])
df = df.drop_duplicates(subset=['prompt', 'response']).reset_index(drop=True)
df['prompt'] = df['prompt'].astype(str).str.strip()
df['response'] = df['response'].astype(str).str.strip()
df = df[df['prompt'].str.len().between(10, 250)]
df = df[df['response'].str.len().between(10, 250)]
df['text'] = df['prompt'] + tokenizer.eos_token + df['response'] + tokenizer.eos_token
print(f"Training on {len(df)} ultra-clean dialogue pairs.")
dataset = Dataset.from_pandas(df)

Using device: cuda
Training on 65203 ultra-clean dialogue pairs.


In [ ]:
def tokenize_function(examples):
    return tokenizer(
        examples["text"], 
        truncation=True, 
        max_length=128
    )
print("Tokenizing dataset...")
tokenized_dataset = dataset.map(
    tokenize_function, 
    batched=True, 
    remove_columns=dataset.column_names 
)
data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

Tokenizing dataset...


Map:   0%|          | 0/65203 [00:00<?, ? examples/s]

In [ ]:
print("Loading pre-trained DialoGPT model in Full Precision...")
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, torch_dtype=torch.float32)
OUTPUT_DIR = "./models/dialogpt-finetuned"
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=3,
    per_device_train_batch_size=4, 
    gradient_accumulation_steps=4,
    learning_rate=1e-5,          
    warmup_steps=200,            # Gently ramps up to avoid shocking the weights
    fp16=False,                  # Must stay False to avoid the 'unscale' error
    logging_steps=50,
    save_steps=5000,
    prediction_loss_only=True,
    report_to="none"
)
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset,
    data_collator=data_collator,
)
# Manually kill the scaler to stop the 'Attempting to unscale FP16' crash
trainer.scaler = None

Loading pre-trained DialoGPT model in Full Precision...


Loading weights:   0%|          | 0/149 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie transformer.wte.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
GPT2LMHeadModel LOAD REPORT from: microsoft/DialoGPT-small
Key                              | Status     |  | 
---------------------------------+------------+--+-
transformer.h.{0...11}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [ ]:
print("Starting training loop...")
torch.cuda.empty_cache()
model.to(torch.float32)
trainer.train()
print(f"Saving final model to {OUTPUT_DIR}...")
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

print("Model Saved!")

Starting training loop...


Step,Training Loss
50,5.261954
100,4.897457
150,4.444477
200,4.108051
250,3.873376
300,3.741638
350,3.600089
400,3.558864
450,3.502455
500,3.479090


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Saving final model to ./models/dialogpt-finetuned...


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model Saved!
